# 🚀 Fine-Tuning Snippy: In-Browser Code Snippet & Tool Calling AI Agent with Gemma 3 (270M)

This notebook demonstrates how to fine-tune **Gemma 3 270M** (`unsloth/gemma-3-270m-it`) as **Snippy** — an in-browser code snippet generator agent. Snippy dynamically selects generic browser tools (`set_background_color`, `show_notification`, `create_ui_element`, `run_javascript`) and executes them live on the webpage via **LiteRT.js** and **WebGPU**.

### Why Gemma 3 270M?
- **Ultra-Compact (~150MB INT8 bundle)**: Downloads and compiles in WebGPU in under a second.
- **Lightning-Fast On-Device Inference**: Instantaneous token generation directly in client browsers with LiteRT.js.
- **Ungated Mirror**: `unsloth/gemma-3-270m-it` runs out of the box without requiring HF tokens.

## Step 1: Install Dependencies
Use `%pip` magic (works in Colab & Jupyter) or `!uv pip install` if using an Astral `uv` environment.

In [ ]:
# In Google Colab or standard Jupyter kernel:
%pip install -q torch transformers peft trl datasets litert-torch litert-lm

# If using Astral uv locally in terminal / notebook:
# !uv pip install -q torch transformers peft trl datasets litert-torch litert-lm

## Step 2: Fine-Tune Gemma 3 270M as 'Snippy' with LoRA / PEFT

We load `unsloth/gemma-3-270m-it` and fine-tune it on Snippy's generic tool-calling dataset.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# Using Gemma 3 270M Instruction-Tuned model!
MODEL_ID = "unsloth/gemma-3-270m-it"
OUTPUT_LORA_DIR = "./lora_adapter"
OUTPUT_MERGED_DIR = "./fine_tuned_gemma_merged"

# 1. Load Tokenizer & Base Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# 2. Configure LoRA for Gemma 3
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
peft_model = get_peft_model(model, peft_config)

# 3. Snippy In-Browser Generic Tool Calling Dataset
sample_data = [
    {
        "text": (
            "<start_of_turn>user\n"
            "Hi, who are you?<end_of_turn>\n"
            "<start_of_turn>model\n"
            "I am Snippy, your in-browser Code Snippet & Tool Calling Agent! Powered by Gemma 3 270M, LiteRT.js, and WebGPU, I execute UI actions directly on this webpage.\n"
            "```js\nSnippy.executeTool('show_notification', { message: 'Snippy Agent is ready to control your browser UI!', type: 'info' });\n```<end_of_turn>"
        )
    },
    {
        "text": (
            "<start_of_turn>user\n"
            "Snippy, make the background dark purple.<end_of_turn>\n"
            "<start_of_turn>model\n"
            "Changing page background color to dark purple!\n"
            "```js\nSnippy.executeTool('set_background_color', { color: '#1e1b4b' });\nSnippy.executeTool('show_notification', { message: 'Background updated to Dark Purple 🌙', type: 'info' });\n```<end_of_turn>"
        )
    },
    {
        "text": (
            "<start_of_turn>user\n"
            "Snippy, create a button called Explore Edge AI.<end_of_turn>\n"
            "<start_of_turn>model\n"
            "Creating an interactive button on the UI!\n"
            "```js\nSnippy.executeTool('create_ui_element', { tag: 'button', text: 'Explore Edge AI', css: 'background: linear-gradient(135deg, #6366f1, #ec4899); color: white; padding: 8px 16px; border-radius: 8px; border: none;', action: \"Snippy.executeTool('show_notification', { message: 'Button clicked!', type: 'success' })\" });\n```<end_of_turn>"
        )
    },
    {
        "text": (
            "<start_of_turn>user\n"
            "Snippy, celebrate with confetti!<end_of_turn>\n"
            "<start_of_turn>model\n"
            "🎉 Celebrating on-device WebGPU AI with confetti!\n"
            "```js\nSnippy.executeTool('run_javascript', { code: 'if (window.confetti) confetti({ particleCount: 100, spread: 70 });' });\nSnippy.executeTool('show_notification', { message: '🎉 Gemma 3 270M WebGPU AI in action!', type: 'success' });\n```<end_of_turn>"
        )
    }
]
dataset = Dataset.from_list(sample_data)

# 4. Training with SFTTrainer
sft_config = SFTConfig(
    dataset_text_field="text",
    max_length=256,
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    logging_steps=1,
    loss_type="nll"
)
trainer = SFTTrainer(
    model=peft_model,
    train_dataset=dataset,
    args=sft_config,
)
trainer.train()

# Save LoRA Adapter
peft_model.save_pretrained(OUTPUT_LORA_DIR)
tokenizer.save_pretrained(OUTPUT_LORA_DIR)
print("✅ Snippy Gemma 3 270M LoRA Adapter Saved!")

## Step 3: Merge LoRA Adapter into Gemma 3 270M Base Model
Unload and merge Snippy's adapter weights into base weights.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_ID = "unsloth/gemma-3-270m-it"
OUTPUT_LORA_DIR = "./lora_adapter"
OUTPUT_MERGED_DIR = "./fine_tuned_gemma_merged"

# Load base model & adapter, then merge
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32, device_map="cpu")
peft_model = PeftModel.from_pretrained(base_model, OUTPUT_LORA_DIR)
merged_model = peft_model.merge_and_unload()

# Save Merged Checkpoint
merged_model.save_pretrained(OUTPUT_MERGED_DIR)
tokenizer.save_pretrained(OUTPUT_MERGED_DIR)
print("✅ Merged Snippy Gemma 3 270M Model Saved to:", OUTPUT_MERGED_DIR)

## Step 4: Convert Model to LiteRT Bundle (`.litertlm`)

Export model to LiteRT bundle format using `litert-torch export_hf` with `-b True`.

In [ ]:
!litert-torch export_hf \
  ./fine_tuned_gemma_merged \
  ./litert_output \
  -b True \
  -q dynamic_int8

## Step 5: Test Snippy with `litert-lm` Native Engine
Verify Snippy's generated JavaScript actions native response.

In [ ]:
import litert_lm

# Initialize Engine with .litertlm bundle
engine = litert_lm.Engine('./litert_output/model.litertlm')
conv = engine.create_conversation()

res = conv.send_message('Snippy, change background color to dark purple.')
print("Snippy Response:", res['content'][0]['text'])

## Step 6: Deploy to Web Browser via LiteRT.js

Copy the exported model bundle and extract the TFLite FlatBuffer to the web server's static assets directory.

In [ ]:
# Copy Converted Model Files to Local Web Demo Directory
import os
import shutil

SOURCE_MODEL = "./litert_output/model.litertlm"
DEST_DIR = "./web/public/models"

os.makedirs(DEST_DIR, exist_ok=True)
if os.path.exists(SOURCE_MODEL):
    # 1. Copy .litertlm bundle for Python backend engine
    shutil.copy(SOURCE_MODEL, os.path.join(DEST_DIR, "model.litertlm"))

    # 2. Extract TFLite FlatBuffer (starts with TFL3 magic) for WebGPU browser runtime
    with open(SOURCE_MODEL, "rb") as f:
        data = f.read()
    pos = data.find(b"TFL3")
    if pos != -1:
        tflite_data = data[pos - 4 :]
        with open(os.path.join(DEST_DIR, "model.tflite"), "wb") as f:
            f.write(tflite_data)
        print("✅ Extracted TFLite FlatBuffer and copied model files to web/public/models/")
    else:
        shutil.copy(SOURCE_MODEL, os.path.join(DEST_DIR, "model.tflite"))
        print("✅ Copied model files to web/public/models/")
else:
    print(f"⚠️ Source model not found at {SOURCE_MODEL}. Make sure Step 4 export finished.")